# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Alien-is-here/FlyRank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

**Answer:**

The Random Forest is used to rank pages by their measured probability of observed decline.

The highest-ranked pages are treated as **review-first candidates**, not guaranteed fixes. Reason codes are added using the observed feature values so that a human reviewer can understand why a page was prioritized.

The main reason signals are:
- lower CTR
- higher average position
- lower engagement rate

The queue is intended to help a human decide which pages to inspect first.

In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_csv("/content/content_refresh_anonymized.csv")
df.head()

,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [6]:
feature_cols = [
    "ctr",
    "avg_position",
    "engagement_rate"
]

X = df[feature_cols]

y = (df["trend_direction"] == "down").astype(int)

groups = df[["client_id","content_id"]]

In [7]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42
)

In [8]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# Group pages by client
gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=df["client_id"])
)

X_train_group = X.iloc[train_idx]
X_test_group = X.iloc[test_idx]

y_train_group = y.iloc[train_idx]
y_test_group = y.iloc[test_idx]

# Train the same Random Forest
rf_group = RandomForestClassifier(
    n_estimators=100,
    random_state=42,
    class_weight="balanced"
)

rf_group.fit(X_train_group, y_train_group)

# Rank test pages by probability of decline
y_prob_group = rf_group.predict_proba(X_test_group)[:, 1]

ranking = pd.DataFrame({
    "actual": y_test_group.values,
    "score": y_prob_group
})

ranking = ranking.sort_values(
    "score",
    ascending=False
)

# Top 50 pages
top_50 = ranking.head(50)

precision_at_50_after = top_50["actual"].mean()

print("Before Precision@50:", 0.740)
print("After Precision@50:", precision_at_50_after)
print("True declines in top 50:", int(top_50["actual"].sum()))

Before Precision@50: 0.74
After Precision@50: 0.62
True declines in top 50: 31


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

**Answer:**

This queue is intended for SEO/content teams to prioritize pages for human review.

The model provides **directional decision-support** by ranking pages that appear more likely to be associated with observed decline in the evaluated data.

The queue should not be interpreted as:
- proof that a page will decline
- proof that a recommended change will improve performance
- proof that refreshing a page causes recovery
- a guarantee of performance on unseen clients or future data

The grouped-by-client audit measured Precision@50 of 0.62, compared with 0.74 under the earlier random split. This shows that performance depends on the validation design and supports using the queue as a prioritization tool rather than an automatic decision-maker.

In [11]:
print("Top-50 review queue")
print("Precision@50:", top_50["actual"].mean())
print("True observed declines:", int(top_50["actual"].sum()))
print("Queue size:", len(top_50))

Top-50 review queue
Precision@50: 0.62
True observed declines: 31
Queue size: 50


In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

**Answer:**

A human should review each high-priority page before taking action.

The reviewer should check:
- whether the observed performance decline is meaningful
- the page's current search performance
- whether the page has recently been updated
- whether the content is still relevant to the intended search need
- whether there are technical or measurement issues that could explain the observed signals

### No-go list

The model should not automatically:
- publish or rewrite content
- delete pages
- change URLs
- make claims about search-engine ranking causes
- assume that a refresh will recover traffic
- make irreversible SEO decisions without human review

The model is a prioritization aid, not an autonomous content decision-maker.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [12]:
# Check that the queue contains only ranking/action information
# and does not expose private client information.

export_columns = [
    "content_id",
    "decline_score",
    "ctr",
    "avg_position",
    "engagement_rate",
    "reason_code",
    "action"
]

print("Export columns:")
print(export_columns)

assert "client_id" not in export_columns
assert "trend_direction" not in export_columns
assert "trend_pct" not in export_columns

print("No client identifier or target-derived field included in the action export.")

Export columns:
['content_id', 'decline_score', 'ctr', 'avg_position', 'engagement_rate', 'reason_code', 'action']
No client identifier or target-derived field included in the action export.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

**Answer:**

The recommendations may become stale if the underlying data or content behavior changes.

The model should be reviewed or retrained when:
- new labeled performance data becomes available
- the distribution of CTR, average position, or engagement rate changes substantially
- Precision@50 decreases consistently on new evaluation data
- the relationship between model scores and observed declines changes
- the content population or measurement process changes

Monitoring should use the same honest validation principle used in this audit, especially grouped-by-client or time-aware evaluation when appropriate.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [14]:
# Current audit baseline for future monitoring

monitoring_baseline = {
    "validation": "grouped_by_client",
    "precision_at_50": top_50["actual"].mean(),
    "true_declines_top_50": int(top_50["actual"].sum()),
    "queue_size": len(top_50)
}

print("Monitoring baseline:")
for key, value in monitoring_baseline.items():
    print(f"{key}: {value}")

Monitoring baseline:
validation: grouped_by_client
precision_at_50: 0.62
true_declines_top_50: 31
queue_size: 50


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

**Answer:**


The top-50 review queue is exported as a CSV for reuse in the paper and for inspection of the model's ranked recommendations.

The export contains pseudonymized content identifiers, model scores, feature values, reason codes, and the suggested review action. Client identifiers and target-derived fields are excluded.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [16]:
import os

output_dir = "work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "ml10_top50_action_queue.csv"
)

# Construct top_50_actions DataFrame with all required columns
# Start with the content_id and feature columns for the top_50 ranked items
top_50_actions_data = df.loc[top_50.index, ['content_id'] + feature_cols].copy()

# Add the decline score (model score) from the top_50 DataFrame
top_50_actions_data['decline_score'] = top_50['score']

# Generate reason codes based on feature values. These thresholds are examples and can be adjusted.
def generate_reason_code(row):
    reasons = []
    # The problem description hints at: "lower CTR", "higher average position", "lower engagement rate"
    # Let's use some example thresholds to create these reasons.
    if row['ctr'] < df['ctr'].quantile(0.25): # e.g., bottom 25% of CTR
        reasons.append('lower CTR')
    if row['avg_position'] > df['avg_position'].quantile(0.75): # e.g., top 25% of avg_position
        reasons.append('higher average position')
    if row['engagement_rate'] < df['engagement_rate'].quantile(0.25): # e.g., bottom 25% of engagement_rate
        reasons.append('lower engagement rate')
    if not reasons:
        return 'general decline score'
    return ', '.join(reasons)

top_50_actions_data['reason_code'] = top_50_actions_data.apply(generate_reason_code, axis=1)

# Add a default action for these top-ranked pages
top_50_actions_data['action'] = 'Review page for potential decline'

# Select and reorder columns according to export_columns
top_50_actions = top_50_actions_data[export_columns]

top_50_actions.to_csv(
    output_path,
    index=False
)

print("Saved:", output_path)
print("Rows:", len(top_50_actions))

Saved: work/outputs/ml10_top50_action_queue.csv
Rows: 50


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.